# Klasyfikacja – wykrywanie oszustw finansowych

W tym notebooku przeprowadzimy EDA, wytrenujemy SVM i porównamy go z innymi klasyfikatorami. Dane pochodzą z Kaggle (Credit Card Fraud Detection 2023). Zmienne V1–V28 są zanonimizowanymi składowymi PCA, więc nie znamy ich pierwotnego znaczenia. `Amount` oznacza kwotę transakcji, a `Class` jest etykietą (0 – prawidłowa transakcja, 1 – oszustwo). Umieść plik `creditcard_2023.csv` w katalogu notebooka.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D
from sklearn.svm import SVC
from joblib import dump, load
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, matthews_corrcoef
from sklearn.model_selection import train_test_split
from time import perf_counter
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
df = pd.read_csv('creditcard_2023.csv')
df.head()

In [ ]:
df = df.drop(columns = ['id'])

In [ ]:
rows, columns = df.shape
print(f"Dane zawierają {columns} kolumn oraz {rows} wierszy.")

In [ ]:
missing_values = df.isnull().sum()
if missing_values[missing_values > 0].sum() > 0:
    print(f"Dane zawierają następującą ilość brakujących wartości:\n{missing_values[missing_values > 0]}")
else:
    print("Dane nie zawierają brakujących wartości.")

In [ ]:
classes=df['Class'].value_counts()
print('Binarne etykiety klas: 1 oznacza oszustwo transakcyjne, a 0 prawidłową transakcję.\n\nLiczebność poszczególnych klas:\n', classes)

### Standaryzacja do wizualizacji

Poniższa standaryzacja służy wyłącznie EDA. Do treningu dopasujemy osobny scaler wyłącznie na danych treningowych.

In [ ]:
scaler_eda = StandardScaler()
scaled = scaler_eda.fit_transform(df.drop(columns=['Class']))
df_scaled = pd.DataFrame(scaled, columns=df.drop(columns=['Class']).columns, index=df.index)
df_scaled

## Podstawowa analiza danych

In [ ]:
df_scaled.describe().round(2)

In [ ]:
df_scaled.hist(bins=50, figsize=(20, 15), color='purple', edgecolor='black')
plt.show()

In [ ]:
plt.figure(figsize=(20, 15))
sns.boxplot(data=df_scaled, orient="h")

plt.title('Wykresy pudełkowe wszystkich cech')
plt.xlabel('Wartość')
plt.ylabel('Cecha')

plt.tight_layout()
plt.show()

### Porównanie rozkładów cech między klasami

Sprawdź, które składowe różnią się rozkładem między transakcjami prawidłowymi i oszustwami.

In [ ]:
df_to_stats = df.drop(columns=["Amount", "Class"])
figsize = (15, 30)
n_cols = 3

n_rows = (len(df_to_stats.columns) + n_cols - 1) // n_cols
fig, axs = plt.subplots(n_rows, n_cols, figsize=figsize)
for i, col in enumerate(df_to_stats.columns):
    row = i // n_cols
    col_idx = i % n_cols
    bp = axs[row, col_idx].boxplot([df.loc[df['Class'] == 0, col], df.loc[df['Class'] == 1, col]], patch_artist=True)
    for box in bp['boxes']:
        box.set(facecolor='lightblue')
    axs[row, col_idx].set_title(f"Wykres pudełkowy dla {col}")
    axs[row, col_idx].set_xticks([1, 2], ["0", "1"])
    axs[row, col_idx].set_xlabel("Klasa")
    axs[row, col_idx].set_ylabel(col)

for ax in axs.flat[len(df_to_stats.columns):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.pie(classes, labels=classes.index, autopct='%1.1f%%', colors=['lightblue', 'purple'])
plt.title('Rozkład wartości dla klas')
plt.show()

In [ ]:
corr_matrix = df.corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm')
plt.show()

### Wizualizacja składowych PCA

Wyświetlamy istniejące składowe V1–V3, nie wykonujemy ponownie PCA. Dla czytelności wykorzystujemy losową próbkę danych.

In [ ]:
plot_df = df_scaled.sample(n=min(10000, len(df_scaled)), random_state=42)
x_axis = plot_df["V1"]
y_axis = plot_df["V2"]

plt.figure(figsize=(8, 6))
plt.scatter(x_axis, y_axis, c=df.loc[plot_df.index, 'Class'], cmap='viridis')
plt.xlabel('V1')
plt.ylabel('V2')
plt.title('Wykres rozproszenia V1 i V2')
plt.show()

In [ ]:
z_axis = plot_df["V3"]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x_axis, y_axis, z_axis, c=df.loc[plot_df.index, 'Class'], cmap='viridis', alpha=0.3)
ax.set_xlabel('V1')
ax.set_ylabel('V2')
ax.set_zlabel('V3')
plt.title('Wykres 3D: V1, V2, V3')
plt.show()

## Budowa i trening modelu

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

# Próbka warsztatowa – zwiększ N, jeśli masz więcej czasu na trening!!!
N = 20000
if len(X) > N:
    X, _, y, _ = train_test_split(X, y, train_size=N, random_state=42, stratify=y)

# 60% trening, 20% walidacja, 20% test
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_raw, y_train, test_size=0.25, random_state=42, stratify=y_train)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)


### Trening SVM

Model liniowy stanowi punkt odniesienia dla pozostałych klasyfikatorów. `C` kontroluje siłę kary za naruszenia marginesu.

Czas treningu i predykcji mierzymy osobno za pomocą `perf_counter()` (w sekundach). Nie wliczamy czasu zapisu modelu ani obliczania metryk.

In [ ]:
svm_model = SVC(kernel='linear', C=1, random_state=42)

start = perf_counter()
svm_model.fit(X_train, y_train)
svm_fit_time = perf_counter() - start

start = perf_counter()
y_pred = svm_model.predict(X_val)
svm_predict_time = perf_counter() - start

print(f'Czas treningu: {svm_fit_time:.3f} s')
print(f'Czas predykcji: {svm_predict_time:.3f} s')

# Zapis modelu razem ze scalerem
dump({'model': svm_model, 'scaler': scaler}, 'svm_model.joblib')

## Ewaluacja modelu na zbiorze walidacyjnym

Zbiór walidacyjny służy do porównania konfiguracji i wybrania tej najlepszej. Zbiór testowy pozostawiamy do końcowej, niezależnej oceny.

In [ ]:
cm = confusion_matrix(y_val, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Klasa przewidywana')
plt.ylabel('Klasa rzeczywista')
plt.show()

In [ ]:
print(f"Accuracy: {np.round(accuracy_score(y_val, y_pred), 5)}")

In [ ]:
print(classification_report(y_val, y_pred))

In [ ]:
print(f"MCC: {np.round(matthews_corrcoef(y_val, y_pred), 5)}")

In [ ]:
class_stats = classification_report(y_val, y_pred, output_dict=True)

print(f"Klasa 0:")
print(f" Precision: {np.round(class_stats['0']['precision'], 5)}")
print(f" Recall: {np.round(class_stats['0']['recall'], 5)}")
print(f" F1-score: {np.round(class_stats['0']['f1-score'], 5)}")
print(f" Support: {np.round(class_stats['0']['support'], 5)}\n")

print(f"Klasa 1:")
print(f" Precision: {np.round(class_stats['1']['precision'], 5)}")
print(f" Recall: {np.round(class_stats['1']['recall'], 5)}")
print(f" F1-score: {np.round(class_stats['1']['f1-score'], 5)}")
print(f" Support: {np.round(class_stats['1']['support'], 5)}")

In [ ]:
pd.DataFrame({'Wartość faktyczna': y_val,'Wartość predykcyjna': y_pred})

## Zadania do samodzielnego wykonania

Celem jest porównanie różnych klasyfikatorów w zadaniu wykrywania oszustw finansowych. Korzystaj z tego samego podziału danych dla wszystkich modeli. Parametry porównuj na przygotowanym zbiorze walidacyjnym (`X_val`, `y_val`). Zbioru testowego użyj dopiero w zadaniu 5.

### Funkcja pomocnicza do porównywania modeli

Funkcja trenuje model, mierzy czas, wykonuje predykcję i zapisuje wyniki. Dla drzew ustaw `dane_surowe=True`. Każda unikalna nazwa tworzy osobny wiersz w tabeli. Czas zależy od sprzętu i ustawień równoległości (Random Forest korzysta z `n_jobs=-1`).

In [ ]:
def metryki(nazwa, y_true, y_pred, fit_time, predict_time):
    return {
        'Model': nazwa,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision (1)': precision_score(y_true, y_pred, zero_division=0),
        'Recall (1)': recall_score(y_true, y_pred, zero_division=0),
        'F1 (1)': f1_score(y_true, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'FP': confusion_matrix(y_true, y_pred, labels=[0, 1])[0, 1],
        'Trening [s]': fit_time,
        'Predykcja [s]': predict_time,
        'Łącznie [s]': fit_time + predict_time
    }

# Zachowujemy wyniki i wytrenowane modele.
wyniki_walidacja = {'SVM linear': metryki('SVM linear', y_val, y_pred,
                                        svm_fit_time, svm_predict_time)}
modele = {'SVM linear': (svm_model, False, svm_fit_time)}

def porownaj_model(nazwa, model, dane_surowe=False):
    X_tr = X_train_raw if dane_surowe else X_train
    X_v = X_val_raw if dane_surowe else X_val

    start = perf_counter()
    model.fit(X_tr, y_train)
    fit_time = perf_counter() - start

    start = perf_counter()
    pred = model.predict(X_v)
    predict_time = perf_counter() - start

    wyniki_walidacja[nazwa] = metryki(nazwa, y_val, pred, fit_time, predict_time)
    modele[nazwa] = (model, dane_surowe, fit_time)
    return pd.DataFrame([wyniki_walidacja[nazwa]]).round(5)

### Zadanie 1. SVM – funkcje jądra

Przetestuj inne funkcje jądra: `rbf`, `poly` oraz `sigmoid`. Porównaj wyniki z jądrem liniowym. Zmień również parametr `C` (np. 0.1, 1, 10).

**Wskazówka:** `C` kontroluje karę za błędy klasyfikacji, `gamma` wpływa na zasięg oddziaływania obserwacji dla jąder nieliniowych, a `degree` określa stopień wielomianu dla `poly`.

Użyj funkcji `porownaj_model()` dla każdego jądra. Zmieniaj nazwę konfiguracji, aby zachować wszystkie wyniki.

In [ ]:
svm_rbf = SVC(kernel='rbf', C=1, gamma='scale')

# TODO: odkomentuj i uruchom, następnie przetestuj inne parametry.
# porownaj_model("SVM rbf", svm_rbf, dane_surowe=False)

### Zadanie 2. Drzewo decyzyjne (Decision Tree)

Wytrenuj drzewo decyzyjne. Sprawdź, jak ograniczenie głębokości (`max_depth`, np. 3, 5, 10) wpływa na jakość klasyfikacji na zbiorze treningowym i walidacyjnym.

**Wskazówka:** drzewa nie wymagają standaryzacji danych. Możesz użyć `X_train_raw` i `X_val_raw`.

Użyj funkcji `porownaj_model()` i nadaj każdej konfiguracji unikalną nazwę.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)

# TODO: odkomentuj i uruchom, następnie przetestuj inne parametry.
# porownaj_model("Decision Tree depth=5", tree_model, dane_surowe=True)

### Zadanie 3. Random Forest

Wytrenuj las losowy i sprawdź wpływ liczby drzew (`n_estimators`, np. 50, 100, 200) oraz ich maksymalnej głębokości.

**Wskazówka:** Random Forest łączy predykcje wielu drzew trenowanych z wykorzystaniem losowości próbek i cech.

Użyj funkcji `porownaj_model()` i nadaj każdej konfiguracji unikalną nazwę.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# TODO: odkomentuj i uruchom, następnie przetestuj inne parametry.
# porownaj_model("Random Forest n=100", rf_model, dane_surowe=True)

### Zadanie 4. Gradient Boosting

Wytrenuj model Gradient Boosting. Sprawdź wpływ parametrów `n_estimators` oraz `learning_rate` (np. 0.05, 0.1, 0.2).

**Wskazówka:** kolejne drzewa są uczone tak, aby poprawiać predykcje dotychczasowego zespołu. `learning_rate` określa siłę wkładu kolejnego drzewa.

Użyj funkcji `porownaj_model()` i nadaj każdej konfiguracji unikalną nazwę.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

# TODO: odkomentuj i uruchom, następnie przetestuj inne parametry.
# porownaj_model("Gradient Boosting n=100", gb_model, dane_surowe=True)

### Zadanie 5. Porównanie modeli

Najpierw porównaj wszystkie eksperymenty na zbiorze walidacyjnym. Wybierz konfiguracje na podstawie metryk i czasu obliczeń. Następnie oceń wybrane modele na zbiorze testowym, bez ponownego treningu i bez zmiany hiperparametrów. Precision, recall i F1-score dotyczą klasy 1 (oszustwo).

- Który model najlepiej wykrywa oszustwa (recall)?
- Który generuje najmniej fałszywych alarmów (FP)?
- Który osiąga najwyższy MCC?
- Który model trenuje się najszybciej, a który najszybciej przewiduje?
- Czy poprawa jakości predykcji uzasadnia dłuższy czas obliczeń?
- Czy wyniki ze zbalansowanego zbioru będą reprezentatywne dla rzeczywistych transakcji?

In [ ]:
# Wszystkie przeprowadzone eksperymenty – zbiór walidacyjny
pd.DataFrame(wyniki_walidacja.values()).sort_values('MCC', ascending=False).round(5)

### Końcowa ewaluacja na zbiorze testowym

Uzupełnij listę nazw wybranych konfiguracji dokładnie tak, jak zapisano je w tabeli walidacyjnej. Czas treningu pochodzi z wcześniejszego pomiaru. Czas predykcji jest mierzony ponownie na zbiorze testowym.

In [ ]:
wybrane_modele = ['SVM linear']  # TODO: dodaj nazwy wybranych konfiguracji
wyniki_test = []

for nazwa in wybrane_modele:
    model, dane_surowe, fit_time = modele[nazwa]
    X_t = X_test_raw if dane_surowe else X_test
    start = perf_counter()
    pred = model.predict(X_t)
    predict_time = perf_counter() - start
    wyniki_test.append(metryki(nazwa, y_test, pred, fit_time, predict_time))

    sns.heatmap(confusion_matrix(y_test, pred, labels=[0, 1]), annot=True, fmt='d', cmap='Blues')
    plt.title(nazwa)
    plt.xlabel('Klasa przewidywana')
    plt.ylabel('Klasa rzeczywista')
    plt.show()

pd.DataFrame(wyniki_test).sort_values('MCC', ascending=False).round(5)